In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

In [ ]:
df = pd.read_csv('Scores.csv', sep=';', decimal=',')
print(df.head())

In [ ]:
missing_data = df.isnull().sum()
print("\n--- Missing Values ---")
print(missing_data[missing_data > 0])

In [ ]:
print(df['gender'].unique())
print(df['subject'].unique())

In [ ]:
gender_avg = df.groupby(['gender', 'subject'])['score'].mean().unstack()
print("\n moyennes par genre et subject: ")
print(gender_avg)

In [ ]:
df['score_cat'] = pd.cut(df['score'],
                     bins=[0, 50, 75, 100],
                     labels= [ 'Low (<50)', 'Medium (50-75)', 'High (>75)'],
                     include_lowest=True,
                     ordered=True)
print("\nScore categories distribution:")
print(df['score_cat'].value_counts())

In [ ]:
score_order = ['Low (<50)', 'Medium (50-75)', 'High (>75)']
df['score_cat'] = pd.Categorical(
    df['score_cat'], 
    categories=score_order, 
    ordered=True)

In [ ]:
crosstab = pd.crosstab(
    [df['subject'], df['score_cat']],
    df['gender'],
    margins=True,
    margins_name='Total'

)
print("\n Effectifs absolus:")
print(crosstab)

In [ ]:
crosstab_pr = pd.crosstab(
    [df['subject'], df['score_cat']],
    df['gender'],
    normalize= 'columns'

) * 100
print("\n Effectifs absolus par pourcentage:")
print(crosstab_pr.round(1).astype(str) + '%')

In [ ]:
crosstab_pr1 = pd.crosstab(
    [df['subject'], df['score_cat']],
    df['gender'],
    normalize= 'index'

) * 100
print("\n Effectifs absolus par pourcentage:")
print(crosstab_pr1.round(1).astype(str) + '%')

In [ ]:
plt.style.use('seaborn-v0_8-whitegrid')
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
fig.suptitle('Repartition des scores par genre, matiere et categorie de score',
             fontsize=16, fontweight='bold', y=1.02)
subjects = ['math', 'language']
colors = { 'Low (<50)': 'red', 'Medium (50-75)': 'orange', 'High (>75)': 'green' }
for ax, subject in zip(axes, subjects):
    subset = df[df['subject'] == subject]
    crosstab_pr2 = pd.crosstab(
        subset['score_cat'], 
        subset['gender'], 
        normalize='columns') *100
    crosstab_pr2.reindex(score_order).plot(
        kind='bar', 
        stacked=True, 
        color=[colors[cat] for cat in score_order], 
        ax=ax,
        edgecolor='white',
        width=0.7)
    ax.set_title(f'{subject}', fontsize=14, fontweight='bold')
    ax.set_xlabel('Genre')
    ax.set_ylabel('Pourcentage (%)')
    ax.tick_params(axis='x', rotation=0)
    ax.legend_.remove()
    ax.grid(axis='y', alpha=0.3)
    
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, title='Categorie de score', loc='upper center', ncol=3, frameon=True, bbox_to_anchor=(0.5, -0.02))
plt.tight_layout()
plt.show()

In [ ]:

print("\n" + "═"*80)
print("📏 STATISTIQUES NUMÉRIQUES PAR GENRE ET MATIÈRE")
print("═"*80)

summary_stats = df.groupby(['subject', 'gender'])['score'].agg([
    ('count', 'count'),
    ('mean', 'mean'),
    ('std', 'std'),
    ('median', 'median'),
    ('min', 'min'),
    ('max', 'max')
]).round(2)

print(summary_stats)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Figure 1 -- Q-Q Plots: Normality Check by Subject and Gender',
             fontsize=13, fontweight='bold')
 
for idx, subject in enumerate(['language', 'math']):
    for jdx, gender in enumerate(['female', 'male']):
        ax = axes[idx, jdx]
        data = df_filtered[
            (df_filtered['subject'] == subject) &
            (df_filtered['gender'] == gender)
        ]['score']
        stats.probplot(data, dist="norm", plot=ax)
        ax.set_title(f"{subject.title()} -- {gender.title()} (n={len(data)})",
                     fontsize=11)
        ax.get_lines()[0].set(color='steelblue', markersize=3, alpha=0.6)
        ax.get_lines()[1].set(color='crimson', linewidth=1.5)
 
plt.tight_layout()
plt.savefig('qq_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print("  -> Figure 1 saved: qq_plots.png")
 

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv('Scores.csv', sep=';', decimal=',')
df_filtered = df[df['subject'].isin(['math', 'language'])].copy()

# Create score categories
df_filtered['score_cat'] = pd.cut(
    df_filtered['score'],
    bins=[0, 50, 75, 100],
    labels=['Low (<50)', 'Medium (50-75)', 'High (>75)'],
    include_lowest=True
)

print("\n" + "═"*70)
print("🧪 HYPOTHESIS TESTS: Male vs Female")
print("═"*70)

for subject in ['language', 'math']:
    print(f"\n📘 {subject.upper()}")
    print("─"*60)
    
    # Get scores by gender
    male = df_filtered[(df_filtered['subject']==subject) & 
                       (df_filtered['gender']=='male')]['score']
    female = df_filtered[(df_filtered['subject']==subject) & 
                         (df_filtered['gender']=='female')]['score']
    
    # ─────────────────────────────────────────
    # 🔹 TEST 1: T-TEST (compare averages)
    # ─────────────────────────────────────────
    print(f"\n  📈 T-TEST: Comparing average scores")
    
    # Levene test for equal variances
    lev_stat, p_levene = stats.levene(male, female)
    equal_var = p_levene > 0.05
    variant   = "Student's t-test" if equal_var else "Welch's t-test"
 
    print(f"\n  🔬 Levene's Test (variance homogeneity):")
    print(f"     • F = {lev_stat:.4f} | p = {p_levene:.4f}")
    print(f"     • Equal variances: {'Yes' if equal_var else 'No'}")
    print(f"     -> Using: {variant}")
 
    
    # Format p-value properly
    t_stat, p_ttest = stats.ttest_ind(male, female, equal_var=equal_var)
    df_ttest = len(male) + len(female) - 2
 
    print(f"\n  📈 {variant}: Comparing mean scores")
    print(f"     H₀: μ_female = μ_male")
    print(f"     H₁: μ_female ≠ μ_male  (two-sided, α = 0.05)")
    print(f"     • ♀ Mean: {female.mean():.2f}  |  ♂ Mean: {male.mean():.2f}")
    print(f"     • Difference: {female.mean() - male.mean():+.2f} points")
    print(f"     • t({df_ttest}) = {t_stat:.3f}")
    p_disp = "p < 0.0001" if p_ttest < 0.0001 else f"p = {p_ttest:.4f}"
    print(f"     • {p_disp}")
 
    if p_ttest < 0.05:
        winner = "♀ females" if female.mean() > male.mean() else "♂ males"
        print(f"     ✅ SIGNIFICANT -- {winner} score significantly higher")
    else:
        print(f"     ❌ Not significant -- no evidence of mean difference")
    #cohen's d calculation
    n1, n2   = len(female), len(male)
    pooled   = np.sqrt(((n1-1)*female.var(ddof=1) + (n2-1)*male.var(ddof=1)) / (n1+n2-2))
    cohens_d = (female.mean() - male.mean()) / pooled
    magnitude = ("negligible" if abs(cohens_d) < 0.2 else
                 "small"      if abs(cohens_d) < 0.5 else
                 "medium"     if abs(cohens_d) < 0.8 else "large")
    print(f"     • Cohen's d = {cohens_d:.3f} -> {magnitude} effect size")
 
    # ─────────────────────────────────────────
    # 🔹 TEST 2: CHI² (compare category distribution)
    # ─────────────────────────────────────────
    print(f"\n  📦 CHI²: Comparing score category distribution")
    
    contingency = pd.crosstab(
        df_filtered[df_filtered['subject']==subject]['gender'],
        df_filtered[df_filtered['subject']==subject]['score_cat']
    )
    
    chi2, p_chi2, dof, _ = stats.chi2_contingency(contingency)
    
    print(f"     • Observed table:")
    print(f"       {contingency.to_string().replace(chr(10), chr(10)+'       ')}")
    print(f"     • χ²({dof}) = {chi2:.3f}")
    
    if p_chi2 < 0.0001:
        p_display = "p < 0.0001"
    else:
        p_display = f"p = {p_chi2:.4f}"
    print(f"     • {p_display}")
    
    if p_chi2 < 0.05:
        print(f"     ✅ SIGNIFICANT: Distribution differs by gender")
    else:
        print(f"     ❌ Not significant")
    
    print("\n" + "─"*60)

## Next Step

Now that the data is cleaned, you can proceed to the analysis notebook: [question2.ipynb](question2.ipynb)